# R1b / R2b — l'étape DPO, sur l'adaptateur produit par le SFT

Seconde moitié d'un bras. Le SFT a appris **un format** à partir de démonstrations ; le DPO
apprend **une préférence** à partir de paires. Dans les termes d'InstructGPT, c'est le
passage de l'étape 1 à l'étape 2 — et les deux sont de l'alignement.

## Pourquoi une soumission séparée

Un bras entier dépasse le plafond de session Kaggle. Le SFT a déjà consommé 5,08 h (A3) et
4,70 h (A2) ; le DPO repart donc du **checkpoint** produit, récupéré par un dataset attaché
plutôt que par `/kaggle/working`, qui appartient au kernel qui l'a écrit et n'est pas
lisible depuis un autre.

## Ce que ce run doit trancher

Le SFT a donné sa réponse : les deux backbones ont appris **la même chose** des mêmes
données — descente de 0,518 pour A3 contre 0,567 pour A2, alors que A3 partait 0,95 nat plus
bas. L'écart est une translation constante, que le SFT n'amplifie pas.

**La question qui reste est de savoir si le DPO, lui, l'amplifie.**

⚠️ **Le chiffre à surveiller en premier n'est pas la loss, c'est `rewards/margins`.** Le DPO
optimise un écart de log-probabilités entre réponse choisie et réponse rejetée. Si le modèle
attribue des probabilités quasi indistinctes aux deux, la marge est du bruit et le gradient
n'apprend rien de stable — et une loss plate ne dirait pas si c'est le backbone qui n'aide
pas, ou s'il n'y avait pas assez de signal. Les marges des premiers pas distinguent les deux.

### Réglages Kaggle
Accelerator **T4 x2**, internet activé, datasets `afrique-safety-dpo-code`,
`afrique-safety-dpo-data` et `afrique-safety-dpo-adapters` attachés.

## 0 · Ce qui change d'un bras à l'autre

**Les trois seules lignes à modifier.** Tout le reste est identique entre A2 et A3 — c'est ce
qui rend l'écart attribuable au backbone et à rien d'autre.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
BRAS       = "A3"                                        # A3 = cible, A2 = controle
MODELE     = "McGill-NLP/AfriqueQwen3.5-4B-50Langs"      # A2: "Qwen/Qwen3.5-4B-Base"
ADAPTATEUR = "A3_s42_sft"                                # A2: "A2_s42_sft"
GRAINE     = 42
# ─────────────────────────────────────────────────────────────────────────────
print(f"bras {BRAS} | {MODELE} | adaptateur {ADAPTATEUR} | graine {GRAINE}")

## 1 · Environnement

Versions **épinglées** sur celles de l'image Kaggle : une montée de version silencieuse entre
deux bras changerait le comportement de TRL sans que rien ne le signale, et l'écart A3 − A2
ne serait plus attribuable au seul backbone.

In [ ]:
!pip install -q -U "transformers==5.16.1" "trl==1.12.0" "peft==0.20.0" bitsandbytes accelerate datasets

In [ ]:
import os, sys
from pathlib import Path

# Un seul GPU visible: Trainer active DataParallel des qu'il en voit plusieurs, ce qui
# doublerait silencieusement le nombre de sequences par pas.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


def racine_du_code():
    """Trouver le dossier contenant src/paths.py, ou qu'il soit monte.

    Kaggle ne monte pas tous les datasets a la meme profondeur: un kernel a montre
    /kaggle/input/<slug>/src/..., un autre /kaggle/input/datasets/<slug>/src/... Un chemin
    code en dur avec un repli evalue immediatement leve StopIteration -- une exception qui
    ne dit rien de ce qui manque. Ce piege a deja coute trois runs; on cherche donc, et on
    affiche ce qui est reellement monte quand on ne trouve pas.
    """
    for base in (Path("/kaggle/input"), Path.cwd(), *Path.cwd().parents):
        if not base.exists():
            continue
        for trouve in base.rglob("paths.py"):
            if trouve.parent.name == "src":
                return trouve.parents[1]

    monte = Path("/kaggle/input")
    contenu = "\n".join(f"  {p}" for p in sorted(monte.rglob("*"))[:40]) if monte.exists() \
        else "  (/kaggle/input absent)"
    raise RuntimeError(
        "src/paths.py introuvable. Attacher afrique-safety-dpo-code.\n"
        f"Ce qui est monte sous {monte} :\n{contenu}"
    )


RACINE_CODE = racine_du_code()
sys.path.insert(0, str(RACINE_CODE))
print("code trouve :", RACINE_CODE)

from src.paths import KAGGLE_INPUT, describe, find_in_datasets, resolve_roots

R = resolve_roots()
ROOT, SORTIE = R["code"], R["output"]

# Recherche recursive: Kaggle ne monte pas tous les datasets a la meme profondeur, et un
# glob a profondeur fixe a deja fait perdre trois runs.
racine_adaptateurs = find_in_datasets(f"*/adapters/{ADAPTATEUR}/adapter_config.json")
if racine_adaptateurs is None:
    raise RuntimeError(
        f"adaptateur {ADAPTATEUR} introuvable. Attacher afrique-safety-dpo-adapters.\n"
        + describe(KAGGLE_INPUT, depth=3)
    )
CHEMIN_ADAPTATEUR = racine_adaptateurs / "adapters" / ADAPTATEUR

print("code       :", ROOT)
print("sorties    :", SORTIE)
print("adaptateur :", CHEMIN_ADAPTATEUR)

import json
provenance = json.loads((CHEMIN_ADAPTATEUR / "provenance.json").read_text(encoding="utf-8"))
print("\nprovenance de l'adaptateur :", json.dumps(provenance, ensure_ascii=False))

# Garde-fou: un adaptateur pose sur le mauvais backbone produit du bruit sans lever
# d'erreur. Le fichier declare sa base, on la compare a celle qu'on s'apprete a charger.
declaree = json.loads((CHEMIN_ADAPTATEUR / "adapter_config.json").read_text())["base_model_name_or_path"]
assert declaree == MODELE, f"adaptateur entraine sur {declaree}, pas sur {MODELE}"
print(f"\nbase declaree par l'adaptateur == MODELE : {declaree}")

import torch, transformers, trl, peft
print(f"\n{torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")
print(f"transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")

## 2 · Les paires de préférence

Le socle DPO, en deux morceaux :

| source | rôle | licence |
| :---- | :---- | :---- |
| Uhura `ha_generation` | `best_answer` contre `incorrect_answers[0]` | MIT |
| UbuntuGuard haoussa, axe Honest | PASS contre FAIL | CC BY 4.0 *(annoncée par le papier, non confirmée par l'autrice)* |

**Aucune génération, aucun juge.** Uhura fournit déjà la bonne et les mauvaises réponses ;
UbuntuGuard fournit déjà le verdict. La paire est donc construite sans qu'un modèle ait à
juger un texte haoussa.

**`axis: honest`.** Ventilées par thème, les 128 paires haoussa d'UbuntuGuard se décomposent
en 95 Honest et 26 Harmless. À 26 paires, l'axe Harmless n'est pas entraînable — il devient
évaluable seulement, via AfriHate et TukaBench.

⚠️ **La graine de partition est FIXE et indépendante de `GRAINE`**, exactement comme au SFT.
La graine d'entraînement fait varier l'initialisation LoRA et l'ordre des exemples ; si elle
faisait aussi varier la partition, les trois graines mélangeraient variance d'entraînement et
variance de partition, et l'écart A3 − A2 ne serait plus attribuable.

In [ ]:
import yaml
from datasets import load_dataset

from src.data import (
    build_preference_pairs,
    build_uhura_pairs,
    filter_by_axis,
    load_ubuntuguard_rows,
    split_by_base_stem,
)

config = yaml.safe_load(open(ROOT / "config.yaml"))
config["training"]["seed"] = GRAINE
config["model"]["base_model_name"] = MODELE
config["paths"]["output_dir"] = str(SORTIE / "results") + "/"
config["dpo"]["adapter_path"] = str(CHEMIN_ADAPTATEUR)

LANGUE = "Hausa"
AXE = config["dpo"]["axis"]

# --- Uhura: la bonne reponse contre la premiere incorrecte, deja fournies
uhura = load_dataset("masakhane/uhura-truthfulqa", "ha_generation", split="test")
paires_uhura = build_uhura_pairs(list(uhura), LANGUE)

# --- UbuntuGuard: PASS contre FAIL, restreint au haoussa puis a l'axe Honest
lignes_ug = [r for r in load_ubuntuguard_rows(R["ubuntuguard"]) if r["language"] == LANGUE]
paires_ug = filter_by_axis(build_preference_pairs(lignes_ug), AXE)

paires = paires_uhura + paires_ug
print(f"Uhura        : {len(paires_uhura)} paires")
print(f"UbuntuGuard  : {len(paires_ug)} paires (axe {AXE}, haoussa)")
print(f"socle total  : {len(paires)} paires")

GRAINE_SPLIT = 42          # FIXE, independante de GRAINE -- voir la cellule markdown
tr_dpo, ev_dpo = split_by_base_stem(paires, seed=GRAINE_SPLIT)
print(f"\n-> {len(tr_dpo)} train / {len(ev_dpo)} eval")
assert not ({p["base_stem"] for p in tr_dpo} & {p["base_stem"] for p in ev_dpo})
print("contamination train <-> eval : 0")

In [ ]:
# La partition d'evaluation est sauvegardee AVANT l'entrainement: sans elle, l'adaptateur
# produit serait inevaluable sur la meme partition, et refaire le split plus tard avec une
# autre graine reintroduirait la contamination qu'on vient de verifier.
(SORTIE / "results").mkdir(parents=True, exist_ok=True)
(SORTIE / "results" / f"dpo_eval_stems_split{GRAINE_SPLIT}.json").write_text(
    json.dumps(sorted({p["base_stem"] for p in ev_dpo}), ensure_ascii=False), encoding="utf-8"
)
print(f"partition d'evaluation sauvegardee ({len(ev_dpo)} base_stems, graine {GRAINE_SPLIT})")

## 3 · Entraînement

`run_dpo` charge le backbone **et** pose l'adaptateur SFT dessus. Deux garde-fous sont dans
`load_causal_lm`, et tous deux protègent d'un échec **silencieux** :

- **`is_trainable=True`** — sans lui, l'adaptateur est chargé gelé, le DPO n'entraîne rien et
  rapporte une loss qui bouge à peine. Le run se termine « normalement » avec un résultat
  faux.
- **`peft_config=None` quand un adaptateur est chargé** — sans quoi un second adaptateur
  s'empilerait sur le premier resté gelé, et on mesurerait l'effet du second seul.

Les deux cas sont verrouillés par `tests/test_handoff.py`.

Le run place un checkpoint tous les 10 pas et écrit ses métriques à chaque log : une session
coupée perd au pire quelques minutes, et relancer ce notebook reprend au dernier checkpoint.

In [ ]:
import time

from src.train import run_dpo

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
depart = time.time()
trainer = run_dpo(config, pairs=tr_dpo)
duree = time.time() - depart
vram = torch.cuda.max_memory_allocated() / 1e9

print(f"\nduree : {duree/3600:.2f} h")
print(f"VRAM  : {vram:.2f} Go")

## 4 · Ce que le run laisse derrière lui

**`rewards/margins` est le chiffre à lire en premier**, avant la loss.

Le DPO optimise `log π(choisie) − log π(rejetée)` relativement au modèle de référence gelé.
La marge dit de combien le modèle sépare les deux réponses. Trois lectures :

| ce qu'on voit | ce que ça veut dire |
| :---- | :---- |
| marge qui croît, exactitude > 0,5 | le DPO apprend ; l'écart A3 − A2 sera interprétable |
| marge qui reste près de zéro | **pas assez de signal** — on ne pourra pas distinguer « le backbone n'aide pas » de « 886 paires ne suffisent pas » |
| marge qui croît mais loss plate | vérifier que l'adaptateur est bien entraînable |

C'est le risque que la fiche des sources nomme explicitement : à 886 paires, une absence
d'effet serait ambiguë. Cette métrique lève l'ambiguïté.

In [ ]:
historique = trainer.state.log_history
marges = [(e["step"], e.get("rewards/margins"), e.get("rewards/accuracies"))
          for e in historique if "rewards/margins" in e]

resume = {
    "bras": BRAS,
    "modele": MODELE,
    "adaptateur_sft": str(CHEMIN_ADAPTATEUR),
    "graine": GRAINE,
    "graine_split": GRAINE_SPLIT,
    "paires_dpo": len(tr_dpo),
    "duree_h": round(duree / 3600, 3),
    "vram_crete_go": round(vram, 2),
    "pas": trainer.state.global_step,
    "loss_finale": next(
        (e["loss"] for e in reversed(historique) if "loss" in e), None
    ),
    "marge_initiale": marges[0][1] if marges else None,
    "marge_finale": marges[-1][1] if marges else None,
    "exactitude_recompense_finale": marges[-1][2] if marges else None,
    "adaptateur": str(SORTIE / "results" / "dpo_checkpoints" / "final"),
}
(SORTIE / "results" / f"dpo_resume_{BRAS}_s{GRAINE}.json").write_text(
    json.dumps(resume, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(resume, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

pertes = [(e["step"], e["loss"]) for e in historique if "loss" in e]

figure, axes = plt.subplots(1, 2, figsize=(12, 3.4))
if pertes:
    pas, valeurs = zip(*pertes)
    axes[0].plot(pas, valeurs)
    axes[0].set_title(f"DPO {BRAS} - loss")
if marges:
    pas_m, valeurs_m, exactitudes = zip(*marges)
    axes[1].plot(pas_m, valeurs_m, label="rewards/margins")
    if any(e is not None for e in exactitudes):
        axes[1].plot(pas_m, exactitudes, label="rewards/accuracies")
    axes[1].axhline(0, color="grey", linewidth=.8)
    axes[1].legend()
    axes[1].set_title(f"DPO {BRAS} - marges de recompense")
for a in axes:
    a.set_xlabel("pas"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

if marges:
    print(f"marge : {marges[0][1]:+.4f} -> {marges[-1][1]:+.4f}")
    if abs(marges[-1][1]) < 0.05:
        print("ATTENTION: marge quasi nulle. Le DPO ne separe pas les deux reponses --")
        print("un resultat nul serait ambigu entre 'le backbone n'aide pas' et")
        print("'886 paires ne suffisent pas'. A declarer tel quel.")

In [ ]:
!ls -la {SORTIE}/results/dpo_checkpoints/final/ 2>/dev/null | head -8

---

## Suite

**Pour le bras de contrôle**, changer les trois lignes de la première cellule : `BRAS = "A2"`,
`MODELE = "Qwen/Qwen3.5-4B-Base"`, `ADAPTATEUR = "A2_s42_sft"`. Tout le reste, graines
comprises, doit rester identique.

**Puis E4** : rejouer E1, E2 et E3 sur A2 et A3. C'est là que le claim se mesure.

Une évaluation **post-SFT et pré-DPO** vaut aussi d'être faite, sur les mêmes axes. Sans ce
point intermédiaire, un écart A3 − A2 positif à la fin ne dirait pas *quelle étape* l'a
produit — c'est exactement l'ablation qu'InstructGPT fait entre ses deux étapes.